In [ ]:
# Fase 3 baseline: duck harness (Tufa Labs) en la G4 — validación offline CORTA.
import json, os, pickle, subprocess, sys, sysconfig, time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1","true"}
NOTEBOOK_START = time.time()
os.environ["MPLBACKEND"] = "Agg"
# thinking OFF: el 36% de los tokens generados eran pensamiento (medido
# en produccion, 1784/1784 llamadas) y el banco no ve mejora de precision.
os.environ["LOCAL_ANALYZER_ENABLE_THINKING"] = "0"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"
# CUDA linker path para vLLM/torch en imagen Kaggle
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in ["/usr/local/nvidia/lib64", os.environ.get("LIBRARY_PATH","")] if e)
# Corte de validación offline (minutos) para NO gastar 9h de G4 cuando no es rerun.
OFFLINE_SOFT_MIN = float(os.environ.get("TAAF_OFFLINE_SOFT_MIN", "25"))
WORKING = Path("/kaggle/working"); WORKING.mkdir(parents=True, exist_ok=True)
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)


In [ ]:
# Instalar arc-agi del wheelhouse de la competencia (offline)
COMP_ROOT = None
for dp, dn, _ in os.walk("/kaggle/input"):
    if "arc_agi_3_wheels" in dn:
        COMP_ROOT = Path(dp); break
assert COMP_ROOT, "wheelhouse no encontrado"
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-index",
    "--no-warn-conflicts","--disable-pip-version-check",
    f"--find-links={COMP_ROOT/'arc_agi_3_wheels'}","arc-agi"], stdout=subprocess.DEVNULL)
import arc_agi; print("arc_agi OK")

# Localizar el bundle del solver por su marker
BUNDLE = None
for m in Path("/kaggle/input").rglob("taaf-kaggle-bundle.json"):
    BUNDLE = m.parent; break
assert BUNDLE, "bundle TAAF no encontrado (adjunta thtennant/taaf-kaggle-source-share-fork)"
print("BUNDLE =", BUNDLE)

# Mapear datasets adjuntos a sus mounts
DATASET_SOURCES = ["thtennant/taaf-kaggle-source-share-fork",
                   "driessmit1/arc3-vllm-h100-wheelhouse-v3",
                   "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
def mount(ref):
    o,s = ref.split("/",1)
    for c in (Path("/kaggle/input")/s, Path("/kaggle/input/datasets")/o/s):
        if c.exists(): return str(c)
    return str(Path("/kaggle/input")/s)
paths = {r: (str(BUNDLE) if i==0 else mount(r)) for i,r in enumerate(DATASET_SOURCES)}
env_extra = {"TAAF_KAGGLE_INPUT_PATHS": json.dumps(paths, sort_keys=True),
             "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
             "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps([])}
os.environ.update(env_extra)
SETUP_ENV = WORKING/"taaf_setup_env.json"; SETUP_ENV.write_text(json.dumps(env_extra))
print(paths)


In [ ]:
# Importar repos del bundle y correr setup_commands (instala vLLM, arranca el server)
def source_entries(b):
    out=[]
    for repo in sorted((b/"src").iterdir(), reverse=True):
        for c in (repo/"src", repo):
            if c.is_dir(): out.append(c)
    return out
entries = source_entries(BUNDLE)
for e in entries: sys.path.insert(0, str(e))
pth = Path(sysconfig.get_paths()["purelib"])/"taaf_sources.pth"
pth.write_text("".join(f"{e}\n" for e in entries))

def cmd_env():
    env = os.environ.copy(); env["PYTHON"]=sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"]=str(BUNDLE); env["TAAF_KAGGLE_WORKING_DIR"]=str(WORKING)
    env["TAAF_KAGGLE_SETUP_ENV"]=str(SETUP_ENV)
    env.update({str(k):str(v) for k,v in json.loads(SETUP_ENV.read_text()).items()})
    return env
env = cmd_env()
for c in json.loads((BUNDLE/"setup_commands.json").read_text()):
    print("setup:", c[:80], flush=True)
    subprocess.run(c, shell=True, check=True, cwd=WORKING, env=env)
    env = cmd_env(); os.environ.update(env)
for e in reversed([x for x in os.environ.get("PYTHONPATH","").split(os.pathsep) if x]):
    if e not in sys.path: sys.path.insert(0, e)
print("setup completo")


In [ ]:
# Cargar benchmark + target, jugar (offline recortado / gateway en rerun)
with open(BUNDLE/"deploy_target.pkl","rb") as f: target = pickle.load(f)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION
with open(BUNDLE/"benchmark_initial.pkl","rb") as f: bm = pickle.load(f)
bm.job_dir = WORKING; bm.n_passes = 1; bm.game_weights = None
os.environ.setdefault("RECORDINGS_DIR", str(WORKING/"server_recording"))

# Graft install (base = config v12 de thtennant, que marco 1.17) + schema_helpers:
# precarga helpers de analisis testeados (grid_diff, connected_components,
# action_effect_summary, recent_history) en el sandbox python del agente — el 27B
# reescribe esa plomeria con bugs en cada juego. NUESTRA tesis de feature injection,
# implementada por el autor del fork como graft sin habilitar (WP3).
# goalkeep NO va: marco 0.81 en el set oculto (-0.36 vs v12; ver working notes
# 2026-08-12). Blindado: cualquier fallo -> stock.
# Verificado en CPU local con scripts/smoke_graft_install.py (banner + prelude 8KB).
try:
    from taaf_grafts.composite import install as _graft_install
    _graft_install(bm, flags={"efficiency": True, "retry_guard": True,
                              "shortcircuit": True, "schema_helpers": True})
except Exception as exc:
    print(f"[taaf_grafts] graft failed, running stock: {type(exc).__name__}: {exc}")

# Refuerzo del thinking OFF: si `tool_agent` ya estaba importado, la variable de
# entorno no basta — su valor quedo congelado en el import (misma leccion que el
# proxy #2 con la temperatura). Se parchea el global del modulo tambien.
try:
    import inference.agent.tool_agent as _ta
    _ta._LOCAL_ANALYZER_ENABLE_THINKING = False
    print("THINKING OFF:", _ta._LOCAL_ANALYZER_ENABLE_THINKING,
          "| temp", _ta._LOCAL_ANALYZER_TEMPERATURE)
except Exception as exc:
    print(f"[no_thinking] no pude parchear el global: {type(exc).__name__}: {exc}")

# HIBRIDO SECUENCIAL: el explorador CPU juega ANTES que el LLM en cada juego.
#
# Medido (docs/DESIGN.md 8.22) sobre los 25 juegos locales con regimen de 2 h:
#   LLM 9 niveles | explorador 18 | UNION 21 — y en parte disjuntos: bp35 y sb26
#   los gana solo el LLM (el explorador da cero alli incluso con 40.000 acciones),
#   tu93 y vc33 solo el explorador.
#
# SECUENCIAL, no concurrente: corre antes de `play()`, asi que `seed_initial_history`
# abre el historial del LLM desde el estado resultante. No hay accion obsoleta, ni
# historial con jugadas ajenas, ni carrera sobre el fichero de estado (los tres
# problemas de 8.21).
#
# Probado contra el `taaf.game.Game` REAL en local (scripts/test_hybrid_prelude.py):
# 6 niveles en 12 juegos con 2.000 acciones. Ese test cazo que `available_actions`
# son ints y no nombres — un stub no lo habria visto.
try:
    import base64 as _b64, sys as _sys, types as _types
    _pkg = _types.ModuleType("arc3"); _pkg.__path__ = []
    _sys.modules["arc3"] = _pkg
    for _n, _s in (("features", "IiIiRmVhdHVyZSBlbmdpbmVlcmluZyBwYXJhIGZyYW1lcyBkZSBBUkMtQUdJLTMuCgpMb3MgZnJhbWVzIHNvbiBncmlkcyA2NHg2NCBjb24gY29sb3JlcyAwLi4xNS4gQXF1w60gc2UgY29tcHV0YW46CiAgLSBmZWF0dXJlcyBwb3IgZnJhbWUgKGhpc3RvZ3JhbWEgZGUgY29sb3IsIG9iamV0b3MsIHNpbWV0csOtYXMsIGJvcmRlcywgZW50cm9ww61hKQogIC0gZmVhdHVyZXMgZGUgdHJhbnNpY2nDs24gKHMsIGEsIHMnKTogcMOteGVsZXMgY2FtYmlhZG9zLCBiYm94IGRlbCBjYW1iaW8sCiAgICBkZWx0YXMgcG9yIGNvbG9yIHkgZGV0ZWNjacOzbiBkZSB0cmFzbGFjacOzbiAodmVjdG9yIGRlIG1vdmltaWVudG8pCgpUb2RvIGVuIG51bXB5IHB1cm8gKHNpbiBzY2lweSkgcGFyYSBwb2RlciBjb3JyZXIgb2ZmbGluZSBlbiBLYWdnbGUgc2luIGRlcHMgZXh0cmEuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVxdWUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgT3B0aW9uYWwsIFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKCk5fQ09MT1JTID0gMTYKR1JJRCA9IDY0CiMgRGVzcGxhemFtaWVudG9zIG3DoXhpbW9zIGEgdGVzdGVhciBhbCBkZXRlY3RhciB0cmFzbGFjacOzbiBkZSBvYmpldG9zIGVudHJlIGZyYW1lcy4KTUFYX1NISUZUID0gOAoKCmRlZiBmcmFtZV90b19ncmlkKGZyYW1lOiBBbnkpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJDb252aWVydGUgRnJhbWVEYXRhLmZyYW1lIChsaXN0YSBkZSBncmlkczsgcHVlZGUgdHJhZXIgdmFyaW9zIHBvciBhbmltYWNpw7NuKQogICAgYWwgw7psdGltbyBncmlkIGNvbW8gbnAubmRhcnJheSAoNjQsIDY0KSBpbnQ4LiIiIgogICAgaWYgZnJhbWUgaXMgTm9uZSBvciBsZW4oZnJhbWUpID09IDA6CiAgICAgICAgcmV0dXJuIG5wLnplcm9zKChHUklELCBHUklEKSwgZHR5cGU9bnAuaW50OCkKICAgIGxhc3QgPSBmcmFtZVstMV0KICAgIHJldHVybiBucC5hc2FycmF5KGxhc3QsIGR0eXBlPW5wLmludDgpCgoKZGVmIGNvbm5lY3RlZF9jb21wb25lbnRzKAogICAgZ3JpZDogbnAubmRhcnJheSwgYmFja2dyb3VuZDogT3B0aW9uYWxbaW50XSA9IE5vbmUKKSAtPiBsaXN0W2RpY3Rbc3RyLCBBbnldXToKICAgICIiIkNvbXBvbmVudGVzIGNvbmV4YXMgNC1jb25lY3RhZGFzIGRlIGNlbGRhcyBkZWwgbWlzbW8gY29sb3IgKGlnbm9yYSBlbCBmb25kbykuCgogICAgRGV2dWVsdmUgdW5hIGxpc3RhIGRlIG9iamV0b3M6IGNvbG9yLCBzaXplLCBiYm94ICh5MCwgeDAsIHkxLCB4MSksIGNlbnRyb2lkLgogICAgQkZTIHB1cm8gZW4gcHl0aG9uOiBlbCBncmlkIGVzIDY0eDY0LCBlcyBiYXJhdG8uCiAgICAiIiIKICAgIGgsIHcgPSBncmlkLnNoYXBlCiAgICBpZiBiYWNrZ3JvdW5kIGlzIE5vbmU6CiAgICAgICAgYmFja2dyb3VuZCA9IGludChucC5iaW5jb3VudChncmlkLnJhdmVsKCksIG1pbmxlbmd0aD1OX0NPTE9SUykuYXJnbWF4KCkpCiAgICBzZWVuID0gbnAuemVyb3MoKGgsIHcpLCBkdHlwZT1ib29sKQogICAgb2JqZWN0czogbGlzdFtkaWN0W3N0ciwgQW55XV0gPSBbXQogICAgZm9yIHkgaW4gcmFuZ2UoaCk6CiAgICAgICAgZm9yIHggaW4gcmFuZ2Uodyk6CiAgICAgICAgICAgIGlmIHNlZW5beSwgeF0gb3IgZ3JpZFt5LCB4XSA9PSBiYWNrZ3JvdW5kOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgY29sb3IgPSBpbnQoZ3JpZFt5LCB4XSkKICAgICAgICAgICAgcSA9IGRlcXVlKFsoeSwgeCldKQogICAgICAgICAgICBzZWVuW3ksIHhdID0gVHJ1ZQogICAgICAgICAgICBjZWxscyA9IFtdCiAgICAgICAgICAgIHdoaWxlIHE6CiAgICAgICAgICAgICAgICBjeSwgY3ggPSBxLnBvcGxlZnQoKQogICAgICAgICAgICAgICAgY2VsbHMuYXBwZW5kKChjeSwgY3gpKQogICAgICAgICAgICAgICAgZm9yIG55LCBueCBpbiAoKGN5IC0gMSwgY3gpLCAoY3kgKyAxLCBjeCksIChjeSwgY3ggLSAxKSwgKGN5LCBjeCArIDEpKToKICAgICAgICAgICAgICAgICAgICBpZiAwIDw9IG55IDwgaCBhbmQgMCA8PSBueCA8IHcgYW5kIG5vdCBzZWVuW255LCBueF0gYW5kIGdyaWRbbnksIG54XSA9PSBjb2xvcjoKICAgICAgICAgICAgICAgICAgICAgICAgc2VlbltueSwgbnhdID0gVHJ1ZQogICAgICAgICAgICAgICAgICAgICAgICBxLmFwcGVuZCgobnksIG54KSkKICAgICAgICAgICAgeXMgPSBbY1swXSBmb3IgYyBpbiBjZWxsc10KICAgICAgICAgICAgeHMgPSBbY1sxXSBmb3IgYyBpbiBjZWxsc10KICAgICAgICAgICAgb2JqZWN0cy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImNvbG9yIjogY29sb3IsCiAgICAgICAgICAgICAgICAgICAgInNpemUiOiBsZW4oY2VsbHMpLAogICAgICAgICAgICAgICAgICAgICJiYm94IjogKG1pbih5cyksIG1pbih4cyksIG1heCh5cyksIG1heCh4cykpLAogICAgICAgICAgICAgICAgICAgICJjZW50cm9pZCI6IChmbG9hdChucC5tZWFuKHlzKSksIGZsb2F0KG5wLm1lYW4oeHMpKSksCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgIG9iamVjdHMuc29ydChrZXk9bGFtYmRhIG86IC1vWyJzaXplIl0pCiAgICByZXR1cm4gb2JqZWN0cwoKCmRlZiBfZWRnZV9kZW5zaXR5KGdyaWQ6IG5wLm5kYXJyYXkpIC0+IGZsb2F0OgogICAgIiIiRnJhY2Npw7NuIGRlIHBhcmVzIHZlY2lub3MgKDQtY29ubikgY29uIGNvbG9yZXMgZGlzdGludG9zOiBtaWRlICdlc3RydWN0dXJhJy4iIiIKICAgIGRoID0gZ3JpZFs6LCAxOl0gIT0gZ3JpZFs6LCA6LTFdCiAgICBkdiA9IGdyaWRbMTosIDpdICE9IGdyaWRbOi0xLCA6XQogICAgcmV0dXJuIGZsb2F0KChkaC5zdW0oKSArIGR2LnN1bSgpKSAvIChkaC5zaXplICsgZHYuc2l6ZSkpCgoKZGVmIF9lbnRyb3B5KGNvdW50czogbnAubmRhcnJheSkgLT4gZmxvYXQ6CiAgICBwID0gY291bnRzW2NvdW50cyA+IDBdLmFzdHlwZShucC5mbG9hdDY0KQogICAgcCAvPSBwLnN1bSgpCiAgICByZXR1cm4gZmxvYXQoLShwICogbnAubG9nMihwKSkuc3VtKCkpCgoKZGVmIGdyaWRfZmVhdHVyZXMoZ3JpZDogbnAubmRhcnJheSwgbWF4X29iamVjdHM6IGludCA9IDgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgIiIiRmVhdHVyZXMgZXNjYWxhcmVzIGRlIHVuIGdyaWQgNjR4NjQuIiIiCiAgICBjb3VudHMgPSBucC5iaW5jb3VudChncmlkLnJhdmVsKCksIG1pbmxlbmd0aD1OX0NPTE9SUylbOk5fQ09MT1JTXQogICAgYmFja2dyb3VuZCA9IGludChjb3VudHMuYXJnbWF4KCkpCiAgICBvYmplY3RzID0gY29ubmVjdGVkX2NvbXBvbmVudHMoZ3JpZCwgYmFja2dyb3VuZCkKICAgIGZlYXRzOiBkaWN0W3N0ciwgQW55XSA9IHsKICAgICAgICAiYmFja2dyb3VuZCI6IGJhY2tncm91bmQsCiAgICAgICAgIm5fY29sb3JzIjogaW50KChjb3VudHMgPiAwKS5zdW0oKSksCiAgICAgICAgImNvbG9yX2VudHJvcHkiOiBfZW50cm9weShjb3VudHMpLAogICAgICAgICJlZGdlX2RlbnNpdHkiOiBfZWRnZV9kZW5zaXR5KGdyaWQpLAogICAgICAgICJzeW1faCI6IGZsb2F0KChncmlkID09IGdyaWRbOiwgOjotMV0pLm1lYW4oKSksICAjIHNpbWV0csOtYSBpenF1aWVyZGEtZGVyZWNoYQogICAgICAgICJzeW1fdiI6IGZsb2F0KChncmlkID09IGdyaWRbOjotMSwgOl0pLm1lYW4oKSksICAjIHNpbWV0csOtYSBhcnJpYmEtYWJham8KICAgICAgICAibl9vYmplY3RzIjogbGVuKG9iamVjdHMpLAogICAgfQogICAgZm9yIGMgaW4gcmFuZ2UoTl9DT0xPUlMpOgogICAgICAgIGZlYXRzW2YiY29sb3Jfe2N9Il0gPSBpbnQoY291bnRzW2NdKQogICAgZm9yIGkgaW4gcmFuZ2UobWF4X29iamVjdHMpOgogICAgICAgIGlmIGkgPCBsZW4ob2JqZWN0cyk6CiAgICAgICAgICAgIG8gPSBvYmplY3RzW2ldCiAgICAgICAgICAgIHkwLCB4MCwgeTEsIHgxID0gb1siYmJveCJdCiAgICAgICAgICAgIGZlYXRzW2Yib2Jqe2l9X2NvbG9yIl0gPSBvWyJjb2xvciJdCiAgICAgICAgICAgIGZlYXRzW2Yib2Jqe2l9X3NpemUiXSA9IG9bInNpemUiXQogICAgICAgICAgICBmZWF0c1tmIm9iantpfV9jeSJdLCBmZWF0c1tmIm9iantpfV9jeCJdID0gb1siY2VudHJvaWQiXQogICAgICAgICAgICBmZWF0c1tmIm9iantpfV9oIl0sIGZlYXRzW2Yib2Jqe2l9X3ciXSA9IHkxIC0geTAgKyAxLCB4MSAtIHgwICsgMQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGZlYXRzW2Yib2Jqe2l9X2NvbG9yIl0gPSAtMQogICAgICAgICAgICBmZWF0c1tmIm9iantpfV9zaXplIl0gPSAwCiAgICAgICAgICAgIGZlYXRzW2Yib2Jqe2l9X2N5Il0gPSBmZWF0c1tmIm9iantpfV9jeCJdID0gLTEuMAogICAgICAgICAgICBmZWF0c1tmIm9iantpfV9oIl0gPSBmZWF0c1tmIm9iantpfV93Il0gPSAwCiAgICByZXR1cm4gZmVhdHMKCgpkZWYgX2RldGVjdF90cmFuc2xhdGlvbihwcmV2OiBucC5uZGFycmF5LCBueHQ6IG5wLm5kYXJyYXksIGRpZmY6IG5wLm5kYXJyYXkpIC0+IHR1cGxlW2ludCwgaW50LCBmbG9hdF06CiAgICAiIiJCdXNjYSBlbCBzaGlmdCAoZHksIGR4KSBxdWUgbWVqb3IgZXhwbGljYSBlbCBjYW1iaW8gY29tbyB0cmFzbGFjacOzbi4KCiAgICBTb2xvIG1pcmEgbGEgcmVnacOzbiBjYW1iaWFkYTogc2kgbnh0ID09IHNoaWZ0KHByZXYpIHNvYnJlIGVzYSByZWdpw7NuLCBoYXkKICAgIG1vdmltaWVudG8gZGUgdW4gb2JqZXRvLiBEZXZ1ZWx2ZSAoZHksIGR4LCBzY29yZSkgY29uIHNjb3JlIGVuIFswLCAxXS4KICAgICIiIgogICAgeXMsIHhzID0gbnAubm9uemVybyhkaWZmKQogICAgaWYgbGVuKHlzKSA9PSAwOgogICAgICAgIHJldHVybiAwLCAwLCAwLjAKICAgIGJlc3QgPSAoMCwgMCwgMC4wKQogICAgZm9yIGR5IGluIHJhbmdlKC1NQVhfU0hJRlQsIE1BWF9TSElGVCArIDEpOgogICAgICAgIGZvciBkeCBpbiByYW5nZSgtTUFYX1NISUZULCBNQVhfU0hJRlQgKyAxKToKICAgICAgICAgICAgaWYgZHkgPT0gMCBhbmQgZHggPT0gMDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHN5LCBzeCA9IHlzIC0gZHksIHhzIC0gZHgKICAgICAgICAgICAgb2sgPSAoc3kgPj0gMCkgJiAoc3kgPCBHUklEKSAmIChzeCA+PSAwKSAmIChzeCA8IEdSSUQpCiAgICAgICAgICAgIGlmIG5vdCBvay5hbnkoKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIG1hdGNoID0gZmxvYXQoKG54dFt5c1tva10sIHhzW29rXV0gPT0gcHJldltzeVtva10sIHN4W29rXV0pLm1lYW4oKSkKICAgICAgICAgICAgaWYgbWF0Y2ggPiBiZXN0WzJdOgogICAgICAgICAgICAgICAgYmVzdCA9IChkeSwgZHgsIG1hdGNoKQogICAgcmV0dXJuIGJlc3QKCgpkZWYgdHJhbnNpdGlvbl9mZWF0dXJlcyhwcmV2X2dyaWQ6IG5wLm5kYXJyYXksIG5leHRfZ3JpZDogbnAubmRhcnJheSkgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJGZWF0dXJlcyBkZWwgY2FtYmlvIGVudHJlIGRvcyBmcmFtZXMgY29uc2VjdXRpdm9zLiIiIgogICAgZGlmZiA9IHByZXZfZ3JpZCAhPSBuZXh0X2dyaWQKICAgIG5fY2hhbmdlZCA9IGludChkaWZmLnN1bSgpKQogICAgZmVhdHM6IGRpY3Rbc3RyLCBBbnldID0geyJuX2NoYW5nZWQiOiBuX2NoYW5nZWR9CiAgICBpZiBuX2NoYW5nZWQgPT0gMDoKICAgICAgICBmZWF0cy51cGRhdGUoCiAgICAgICAgICAgIHsiY2hnX3kwIjogLTEsICJjaGdfeDAiOiAtMSwgImNoZ19oIjogMCwgImNoZ193IjogMCwKICAgICAgICAgICAgICJjaGdfYXJlYV9mcmFjIjogMC4wLCAibW92ZV9keSI6IDAsICJtb3ZlX2R4IjogMCwgIm1vdmVfc2NvcmUiOiAwLjAsCiAgICAgICAgICAgICAiY29sb3JzX2dhaW5lZCI6IDAsICJjb2xvcnNfbG9zdCI6IDB9CiAgICAgICAgKQogICAgICAgIHJldHVybiBmZWF0cwogICAgeXMsIHhzID0gbnAubm9uemVybyhkaWZmKQogICAgeTAsIHkxLCB4MCwgeDEgPSB5cy5taW4oKSwgeXMubWF4KCksIHhzLm1pbigpLCB4cy5tYXgoKQogICAgZmVhdHNbImNoZ195MCJdLCBmZWF0c1siY2hnX3gwIl0gPSBpbnQoeTApLCBpbnQoeDApCiAgICBmZWF0c1siY2hnX2giXSwgZmVhdHNbImNoZ193Il0gPSBpbnQoeTEgLSB5MCArIDEpLCBpbnQoeDEgLSB4MCArIDEpCiAgICBmZWF0c1siY2hnX2FyZWFfZnJhYyJdID0gZmxvYXQobl9jaGFuZ2VkIC8gZGlmZi5zaXplKQogICAgcHJldl9jb3VudHMgPSBucC5iaW5jb3VudChwcmV2X2dyaWQucmF2ZWwoKSwgbWlubGVuZ3RoPU5fQ09MT1JTKVs6Tl9DT0xPUlNdCiAgICBuZXh0X2NvdW50cyA9IG5wLmJpbmNvdW50KG5leHRfZ3JpZC5yYXZlbCgpLCBtaW5sZW5ndGg9Tl9DT0xPUlMpWzpOX0NPTE9SU10KICAgIGRlbHRhID0gbmV4dF9jb3VudHMuYXN0eXBlKGludCkgLSBwcmV2X2NvdW50cy5hc3R5cGUoaW50KQogICAgZmVhdHNbImNvbG9yc19nYWluZWQiXSA9IGludCgoZGVsdGEgPiAwKS5zdW0oKSkKICAgIGZlYXRzWyJjb2xvcnNfbG9zdCJdID0gaW50KChkZWx0YSA8IDApLnN1bSgpKQogICAgZHksIGR4LCBzY29yZSA9IF9kZXRlY3RfdHJhbnNsYXRpb24ocHJldl9ncmlkLCBuZXh0X2dyaWQsIGRpZmYpCiAgICBmZWF0c1sibW92ZV9keSJdLCBmZWF0c1sibW92ZV9keCJdLCBmZWF0c1sibW92ZV9zY29yZSJdID0gZHksIGR4LCBzY29yZQogICAgcmV0dXJuIGZlYXRzCgoKZGVmIGFjdGlvbl9lZmZlY3Rfc3VtbWFyeShyb3dzOiBTZXF1ZW5jZVtkaWN0W3N0ciwgQW55XV0pIC0+IGxpc3RbZGljdFtzdHIsIEFueV1dOgogICAgIiIiUmVzdW1lbiBwb3IgYWNjacOzbiBhIHBhcnRpciBkZSBmaWxhcyBkZSB0cmFuc2ljacOzbjogwr9xdcOpIGFjY2lvbmVzICdoYWNlbiBhbGdvJz8KCiAgICBDYWRhIGZpbGEgZGViZSB0cmFlcjogYWN0aW9uX2lkLCBuX2NoYW5nZWQsIGxldmVsX3VwIChib29sKSwgZ2FtZV9vdmVyIChib29sKS4KICAgICIiIgogICAgb3V0OiBsaXN0W2RpY3Rbc3RyLCBBbnldXSA9IFtdCiAgICBieV9hY3Rpb246IGRpY3RbaW50LCBsaXN0W2RpY3Rbc3RyLCBBbnldXV0gPSB7fQogICAgZm9yIHIgaW4gcm93czoKICAgICAgICBieV9hY3Rpb24uc2V0ZGVmYXVsdChpbnQoclsiYWN0aW9uX2lkIl0pLCBbXSkuYXBwZW5kKHIpCiAgICBmb3IgYWN0aW9uX2lkLCBycyBpbiBzb3J0ZWQoYnlfYWN0aW9uLml0ZW1zKCkpOgogICAgICAgIG4gPSBsZW4ocnMpCiAgICAgICAgb3V0LmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImFjdGlvbl9pZCI6IGFjdGlvbl9pZCwKICAgICAgICAgICAgICAgICJuX3VzZXMiOiBuLAogICAgICAgICAgICAgICAgInBfY2hhbmdlIjogZmxvYXQobnAubWVhbihbclsibl9jaGFuZ2VkIl0gPiAwIGZvciByIGluIHJzXSkpLAogICAgICAgICAgICAgICAgImF2Z19waXhlbHNfY2hhbmdlZCI6IGZsb2F0KG5wLm1lYW4oW3JbIm5fY2hhbmdlZCJdIGZvciByIGluIHJzXSkpLAogICAgICAgICAgICAgICAgInBfbGV2ZWxfdXAiOiBmbG9hdChucC5tZWFuKFtib29sKHIuZ2V0KCJsZXZlbF91cCIpKSBmb3IgciBpbiByc10pKSwKICAgICAgICAgICAgICAgICJwX2dhbWVfb3ZlciI6IGZsb2F0KG5wLm1lYW4oW2Jvb2woci5nZXQoImdhbWVfb3ZlciIpKSBmb3IgciBpbiByc10pKSwKICAgICAgICAgICAgfQogICAgICAgICkKICAgIHJldHVybiBvdXQK"), ("agent", "IiIiR3JhcGhFeHBsb3JlcjogYWdlbnRlIGRlIGV4cGxvcmFjacOzbiBkZSBncmFmbyBkZSBlc3RhZG9zIHBhcmEgQVJDLUFHSS0zLgoKU8OtbnRlc2lzIGRlIGxvIG1lam9yIGRlbCBsZWFkZXJib2FyZCBww7pibGljbyAodmVyIGRvY3MvU1RSQVRFR1kubWQpOgogIC0gR3JhZm8gZGUgZXN0YWRvcyBjb24gaGFzaGluZyBlbm1hc2NhcmFkbyAoYm9yZGUgM3B4ICsgbcOhc2NhcmEgZGUgY29udGFkb3IgYXByZW5kaWRhKSwKICAgIEJGUyBzb2JyZSBlbCBncmFmbyBhcHJlbmRpZG8gcGFyYSB2b2x2ZXIgYSBub2RvcyBjb24gYWNjaW9uZXMgcGVuZGllbnRlcywgeSByZXBsYXkKICAgIHRyYXMgUkVTRVQgYXByb3ZlY2hhbmRvIGVsIGRldGVybWluaXNtbyBkZSBsb3MganVlZ29zLiAgW2VzdGlsbyB2NDcsIExCIDAuNTRdCiAgLSBDbGlja3MgcG9yIGNvbXBvbmVudGVzIGNvbmV4YXMgb3JkZW5hZGFzIHBvciBidXR0b24tbGlrZW5lc3MgKGNvbXBhY3RvK3BlcXVlw7FvK2NvbG9yCiAgICByYXJvKSArIHJlamlsbGEgZ3J1ZXNhIGRlIGNvYmVydHVyYTsgc3VwcmVzacOzbiAiZGVhZHNpZyIgZGUgY2xhc2VzIGVzdHJ1Y3R1cmFsbWVudGUKICAgIGluZXJ0ZXMgY29uIHByb3RlY2Npw7NuIGRlIGNsYXNlcyBhbGd1bmEgdmV6IGVmZWN0aXZhcy4gICBbZXN0aWxvIDLCuiBtaWxlc3RvbmVdCiAgLSBPcmRlbiBkZSBhY2Npb25lcyBzaW1wbGVzIHBvciBQKGNhbWJpbykgYXByZW5kaWRhIG9ubGluZTsgbm8tb3BzIHNlIGh1bmRlbiwgbm8gc2UgcG9kYW4uCgpMw7NnaWNhIHB1cmEgc29icmUgbnVtcHk6IGVsIHJ1bm5lciAobG9jYWwgbyBnYXRld2F5KSBsZSBwYXNhIGZyYW1lcyB5IGVqZWN1dGEgbG8gcXVlIGVsaWdlLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlcXVlCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIE9wdGlvbmFsCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmZlYXR1cmVzIGltcG9ydCBjb25uZWN0ZWRfY29tcG9uZW50cwoKR1JJRCA9IDY0CkJPUkRFUiA9IDMgICAgICAgICAgICMgYm9yZGUgZW5tYXNjYXJhZG8gZGVsIGhhc2g6IEhVRC9jb250YWRvcmVzIHZpdmVuIGFow60KQ09VTlRFUl9XQVJNVVAgPSAxMiAgIyB0cmFuc2ljaW9uZXMgcGFyYSBhcHJlbmRlciBsYSBtw6FzY2FyYSBkZSBjb250YWRvcgpDT1VOVEVSX0ZSQUNUSU9OID0gMC44ICAgIyBjZWxkYSBjb250YWRvciBzaSBjYW1iaWEgZW4gPj04MCUgZGUgbGFzIHRyYW5zaWNpb25lcwpDT1VOVEVSX01BWF9JTlRFUklPUiA9IDAuMiAgIyBsYSBtw6FzY2FyYSBhcHJlbmRpZGEgbm8gcHVlZGUgdGFwYXIgPjIwJSBkZWwgaW50ZXJpb3IKQ0xJQ0tfQ0FQID0gNjQgICAgICAgIyBjYW5kaWRhdG9zIGRlIGNsaWNrIHBvciBub2RvCkRFQURfSyA9IDIgICAgICAgICAgICMgY2xhc2UgZGUgY2xpY2sgbXVlcnRhIHRyYXMgSyB1c29zIGluZXJ0ZXMKTUFYX0VYSEFVU1RFRF9SRVNFVFMgPSA0MCAgICMgcmVpbmljaW9zIGRpdmVyc2lmaWNhZG9zIGFudGVzIGRlIHJlbmRpcnNlClJFU0VUX0xPT1BfQlJFQUsgPSAzMAoKIyBpZHMgZGUgYWNjacOzbjogMD1SRVNFVCwgMS4uNSB5IDcgc2ltcGxlcywgNj1jbGljayh4LHkpClNJTVBMRV9JRFMgPSAoMSwgMiwgMywgNCwgNSwgNykKUkVTRVRfS0VZID0gKDAsIC0xLCAtMSkKCgpjbGFzcyBfTm9kZToKICAgIF9fc2xvdHNfXyA9ICgicGVuZGluZyIsICJ0cmllZCIpCgogICAgZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5wZW5kaW5nOiBkZXF1ZVt0dXBsZVtpbnQsIGludCwgaW50XV0gPSBkZXF1ZSgpCiAgICAgICAgc2VsZi50cmllZDogc2V0W3R1cGxlW2ludCwgaW50LCBpbnRdXSA9IHNldCgpCgoKY2xhc3MgR3JhcGhFeHBsb3JlcjoKICAgICIiIkVsaWdlIChhY3Rpb25faWQsIHgsIHkpLiBFbCBjYWxsZXIgZWplY3V0YSB5IGRldnVlbHZlIGVsIGZyYW1lIGVuIGVsIHByw7N4aW1vIGNob29zZSgpLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBnYW1lX2lkOiBzdHIgPSAiIiwgbWF4X2FjdGlvbnM6IGludCA9IDE1MDAwKSAtPiBOb25lOgogICAgICAgIHNlbGYuZ2FtZV9pZCA9IGdhbWVfaWQKICAgICAgICBzZWxmLm1heF9hY3Rpb25zID0gbWF4X2FjdGlvbnMKICAgICAgICBzZWxmLmFjdGlvbnNfdGFrZW4gPSAwCgogICAgICAgIHNlbGYuX25vZGVzOiBkaWN0W2ludCwgX05vZGVdID0ge30KICAgICAgICBzZWxmLl9lZGdlczogZGljdFt0dXBsZVtpbnQsIHR1cGxlW2ludCwgaW50LCBpbnRdXSwgaW50XSA9IHt9CiAgICAgICAgc2VsZi5fYWRqOiBkaWN0W2ludCwgbGlzdFt0dXBsZVt0dXBsZVtpbnQsIGludCwgaW50XSwgaW50XV1dID0ge30KCiAgICAgICAgc2VsZi5fY291bnRlcl9jb3VudHMgPSBucC56ZXJvcygoR1JJRCwgR1JJRCksIGR0eXBlPW5wLmludDMyKQogICAgICAgIHNlbGYuX2NvdW50ZXJfc2VlbiA9IDAKICAgICAgICBzZWxmLl9jb3VudGVyX21hc2s6IE9wdGlvbmFsW25wLm5kYXJyYXldID0gTm9uZQoKICAgICAgICAjIHN0YXRzIHBvciBhY2Npw7NuIHNpbXBsZTogW2NhbWJpb3MsIHVzb3MsIG5vZG9zX251ZXZvc10KICAgICAgICBzZWxmLl9hY3Rfc3RhdHM6IGRpY3RbaW50LCBsaXN0W2ludF1dID0ge2E6IFswLCAwLCAwXSBmb3IgYSBpbiBTSU1QTEVfSURTfQogICAgICAgICMgZGVhZHNpZyBwb3IgY2xhc2UgZXN0cnVjdHVyYWwgZGUgY2xpY2sgKGNvbG9yLCBzaXplLCBpc19yZWN0KQogICAgICAgIHNlbGYuX2RlYWRfc2lnczogZGljdFt0dXBsZVtpbnQsIGludCwgYm9vbF0sIGludF0gPSB7fQogICAgICAgIHNlbGYuX2VmZl9zaWdzOiBzZXRbdHVwbGVbaW50LCBpbnQsIGJvb2xdXSA9IHNldCgpCgogICAgICAgIHNlbGYuX2xhc3Rfa2V5OiBPcHRpb25hbFtpbnRdID0gTm9uZQogICAgICAgIHNlbGYuX2xhc3RfYWN0aW9uOiBPcHRpb25hbFt0dXBsZVtpbnQsIGludCwgaW50XV0gPSBOb25lCiAgICAgICAgc2VsZi5fbGFzdF9ncmlkOiBPcHRpb25hbFtucC5uZGFycmF5XSA9IE5vbmUKICAgICAgICBzZWxmLl9sYXN0X2xldmVscyA9IDAKICAgICAgICBzZWxmLl9yZXBsYXk6IGRlcXVlW3R1cGxlW2ludCwgaW50LCBpbnRdXSA9IGRlcXVlKCkKICAgICAgICBzZWxmLl9yZXBsYXlfdGFyZ2V0OiBPcHRpb25hbFtpbnRdID0gTm9uZQogICAgICAgIHNlbGYuX2V4aGF1c3RlZF9yZXNldHMgPSAwCiAgICAgICAgc2VsZi5fY29uc2VjdXRpdmVfcmVzZXRzID0gMAogICAgICAgIHNlbGYuX3NhbHQgPSAwICAgIyBwZXJ0dXJiYSBlbCBvcmRlbiBkZSBjYW5kaWRhdG9zIGVuIGNhZGEgcmVpbmljaW8gZGl2ZXJzaWZpY2FkbwogICAgICAgIHNlbGYuZG9uZSA9IEZhbHNlCgogICAgIyAtLS0tLS0tLS0tIGhhc2hpbmcgLS0tLS0tLS0tLQoKICAgIGRlZiBfbWFzayhzZWxmKSAtPiBucC5uZGFycmF5OgogICAgICAgIG0gPSBucC56ZXJvcygoR1JJRCwgR1JJRCksIGR0eXBlPWJvb2wpCiAgICAgICAgbVs6Qk9SREVSLCA6XSA9IG1bLUJPUkRFUjosIDpdID0gbVs6LCA6Qk9SREVSXSA9IG1bOiwgLUJPUkRFUjpdID0gVHJ1ZQogICAgICAgIGlmIHNlbGYuX2NvdW50ZXJfbWFzayBpcyBub3QgTm9uZToKICAgICAgICAgICAgbSB8PSBzZWxmLl9jb3VudGVyX21hc2sKICAgICAgICByZXR1cm4gbQoKICAgIGRlZiBfa2V5KHNlbGYsIGdyaWQ6IG5wLm5kYXJyYXkpIC0+IGludDoKICAgICAgICBnID0gZ3JpZC5jb3B5KCkKICAgICAgICBnW3NlbGYuX21hc2soKV0gPSAwCiAgICAgICAgcmV0dXJuIGhhc2goZy50b2J5dGVzKCkpCgogICAgZGVmIF9sZWFybl9jb3VudGVyX21hc2soc2VsZiwgcHJldjogbnAubmRhcnJheSwgbnh0OiBucC5uZGFycmF5KSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYuX2NvdW50ZXJfbWFzayBpcyBub3QgTm9uZSBvciBwcmV2IGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIHNlbGYuX2NvdW50ZXJfY291bnRzICs9IHByZXYgIT0gbnh0CiAgICAgICAgc2VsZi5fY291bnRlcl9zZWVuICs9IDEKICAgICAgICBpZiBzZWxmLl9jb3VudGVyX3NlZW4gPj0gQ09VTlRFUl9XQVJNVVA6CiAgICAgICAgICAgIGNhbmQgPSBzZWxmLl9jb3VudGVyX2NvdW50cyA+PSBDT1VOVEVSX0ZSQUNUSU9OICogc2VsZi5fY291bnRlcl9zZWVuCiAgICAgICAgICAgIGNhbmRbOkJPUkRFUiwgOl0gPSBjYW5kWy1CT1JERVI6LCA6XSA9IGNhbmRbOiwgOkJPUkRFUl0gPSBjYW5kWzosIC1CT1JERVI6XSA9IEZhbHNlCiAgICAgICAgICAgIGludGVyaW9yID0gKEdSSUQgLSAyICogQk9SREVSKSAqKiAyCiAgICAgICAgICAgICMgc2kgInRvZG8gY2FtYmlhIHNpZW1wcmUiIChhbmltYWNpw7NuIGdsb2JhbCkgbGEgbcOhc2NhcmEgc2Vyw61hIGluw7p0aWw6IGJvcmRlIHNvbG8KICAgICAgICAgICAgc2VsZi5fY291bnRlcl9tYXNrID0gY2FuZCBpZiBjYW5kLnN1bSgpIDw9IENPVU5URVJfTUFYX0lOVEVSSU9SICogaW50ZXJpb3IgXAogICAgICAgICAgICAgICAgZWxzZSBucC56ZXJvcygoR1JJRCwgR1JJRCksIGR0eXBlPWJvb2wpCgogICAgIyAtLS0tLS0tLS0tIGNhbmRpZGF0b3MgLS0tLS0tLS0tLQoKICAgIGRlZiBfY2xpY2tfc2lnKHNlbGYsIG9iajogZGljdFtzdHIsIEFueV0pIC0+IHR1cGxlW2ludCwgaW50LCBib29sXToKICAgICAgICB5MCwgeDAsIHkxLCB4MSA9IG9ialsiYmJveCJdCiAgICAgICAgaXNfcmVjdCA9IG9ialsic2l6ZSJdID09ICh5MSAtIHkwICsgMSkgKiAoeDEgLSB4MCArIDEpCiAgICAgICAgcmV0dXJuIChvYmpbImNvbG9yIl0sIG9ialsic2l6ZSJdLCBpc19yZWN0KQoKICAgIGRlZiBfY2xpY2tfY2FuZGlkYXRlcyhzZWxmLCBncmlkOiBucC5uZGFycmF5KSAtPiBsaXN0W3R1cGxlW2ludCwgaW50LCBpbnRdXToKICAgICAgICBjb3VudHMgPSBucC5iaW5jb3VudChncmlkLnJhdmVsKCksIG1pbmxlbmd0aD0xNikKICAgICAgICBiYWNrZ3JvdW5kID0gaW50KGNvdW50cy5hcmdtYXgoKSkKICAgICAgICB0b3RhbCA9IGdyaWQuc2l6ZQogICAgICAgIG9ianMgPSBjb25uZWN0ZWRfY29tcG9uZW50cyhncmlkLCBiYWNrZ3JvdW5kKQogICAgICAgIHNjb3JlZCA9IFtdCiAgICAgICAgZm9yIG8gaW4gb2JqczoKICAgICAgICAgICAgc2lnID0gc2VsZi5fY2xpY2tfc2lnKG8pCiAgICAgICAgICAgIGlmIHNlbGYuX2RlYWRfc2lncy5nZXQoc2lnLCAwKSA+PSBERUFEX0sgYW5kIHNpZyBub3QgaW4gc2VsZi5fZWZmX3NpZ3M6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICByYXJpdHkgPSAxLjAgLSBjb3VudHNbb1siY29sb3IiXV0gLyB0b3RhbAogICAgICAgICAgICB5MCwgeDAsIHkxLCB4MSA9IG9bImJib3giXQogICAgICAgICAgICBmaWxsID0gb1sic2l6ZSJdIC8gKCh5MSAtIHkwICsgMSkgKiAoeDEgLSB4MCArIDEpKQogICAgICAgICAgICBzaXplX3Njb3JlID0gMS4wIGlmIG9bInNpemUiXSA8PSA0IGVsc2UgMC44IGlmIG9bInNpemUiXSA8PSAxNiBlbHNlIFwKICAgICAgICAgICAgICAgIDAuNSBpZiBvWyJzaXplIl0gPD0gNjQgZWxzZSAwLjI1IGlmIG9bInNpemUiXSA8PSAyNTYgZWxzZSAwLjAKICAgICAgICAgICAgc2NvcmUgPSAwLjQgKiByYXJpdHkgKyAwLjMgKiBzaXplX3Njb3JlICsgMC4zICogZmlsbAogICAgICAgICAgICBjeSwgY3ggPSBvWyJjZW50cm9pZCJdCiAgICAgICAgICAgIHNjb3JlZC5hcHBlbmQoKHNjb3JlLCBpbnQocm91bmQoY3gpKSwgaW50KHJvdW5kKGN5KSkpKQogICAgICAgIHNjb3JlZC5zb3J0KGtleT1sYW1iZGEgdDogLXRbMF0pCiAgICAgICAgY2FuZHMgPSBbKDYsIHgsIHkpIGZvciBfLCB4LCB5IGluIHNjb3JlZFs6Q0xJQ0tfQ0FQXV0KICAgICAgICAjIHJlamlsbGEgZGUgY29iZXJ0dXJhOyBzZSBkZW5zaWZpY2EgY29uIGNhZGEgcmVpbmljaW8gZGl2ZXJzaWZpY2FkbyAoc2FsdCkKICAgICAgICBzdHJpZGUgPSA4IGlmIHNlbGYuX3NhbHQgPT0gMCBlbHNlIDQgaWYgc2VsZi5fc2FsdCA8IDMgZWxzZSAyCiAgICAgICAgb2Zmc2V0ID0gKHNlbGYuX3NhbHQgKiAzKSAlIG1heChzdHJpZGUsIDEpCiAgICAgICAgY2FwID0gQ0xJQ0tfQ0FQIGlmIHNlbGYuX3NhbHQgPT0gMCBlbHNlIENMSUNLX0NBUCAqIDQKICAgICAgICBmb3IgZ3kgaW4gcmFuZ2Uob2Zmc2V0LCBHUklELCBzdHJpZGUpOgogICAgICAgICAgICBmb3IgZ3ggaW4gcmFuZ2Uob2Zmc2V0LCBHUklELCBzdHJpZGUpOgogICAgICAgICAgICAgICAgaWYgbGVuKGNhbmRzKSA+PSBjYXA6CiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGMgPSAoNiwgZ3gsIGd5KQogICAgICAgICAgICAgICAgaWYgYyBub3QgaW4gY2FuZHM6CiAgICAgICAgICAgICAgICAgICAgY2FuZHMuYXBwZW5kKGMpCiAgICAgICAgcmV0dXJuIGNhbmRzWzpjYXBdCgogICAgZGVmIF9zaW1wbGVfb3JkZXIoc2VsZiwgYXZhaWxhYmxlOiBsaXN0W2ludF0pIC0+IGxpc3RbaW50XToKICAgICAgICBhY3RzID0gW2EgZm9yIGEgaW4gU0lNUExFX0lEUyBpZiBub3QgYXZhaWxhYmxlIG9yIGEgaW4gYXZhaWxhYmxlXQogICAgICAgICMgcm90YWNpw7NuIHBvciBzYWx0OiBjYWRhIHJlaW5pY2lvIGRpdmVyc2lmaWNhZG8gcHJ1ZWJhIHVuIG9yZGVuIGJhc2UgZGlzdGludG8sCiAgICAgICAgIyBhc8OtIGVsIGRlc2VtcGF0ZSBlbnRyZSBhY2Npb25lcyBubyBwcm9iYWRhcyBnZW5lcmEgdHJheWVjdG9yaWFzIG51ZXZhcy4KICAgICAgICBpZiBzZWxmLl9zYWx0IGFuZCBhY3RzOgogICAgICAgICAgICByID0gc2VsZi5fc2FsdCAlIGxlbihhY3RzKQogICAgICAgICAgICBhY3RzID0gYWN0c1tyOl0gKyBhY3RzWzpyXQoKICAgICAgICBkZWYgc2NvcmUoYTogaW50KSAtPiBmbG9hdDoKICAgICAgICAgICAgY2hnLCB1c2VzLCBfbmV3ID0gc2VsZi5fYWN0X3N0YXRzW2FdCiAgICAgICAgICAgIHJldHVybiAwLjUgaWYgdXNlcyA9PSAwIGVsc2UgY2hnIC8gdXNlcwoKICAgICAgICByZXR1cm4gc29ydGVkKGFjdHMsIGtleT1sYW1iZGEgYTogLXNjb3JlKGEpKQoKICAgIGRlZiBfZmlsbF9wZW5kaW5nKHNlbGYsIG5vZGU6IF9Ob2RlLCBncmlkOiBucC5uZGFycmF5LCBhdmFpbGFibGU6IGxpc3RbaW50XSkgLT4gTm9uZToKICAgICAgICBmb3IgYSBpbiBzZWxmLl9zaW1wbGVfb3JkZXIoYXZhaWxhYmxlKToKICAgICAgICAgICAgayA9IChhLCAtMSwgLTEpCiAgICAgICAgICAgIGlmIGsgbm90IGluIG5vZGUudHJpZWQ6CiAgICAgICAgICAgICAgICBub2RlLnBlbmRpbmcuYXBwZW5kKGspCiAgICAgICAgaWYgbm90IGF2YWlsYWJsZSBvciA2IGluIGF2YWlsYWJsZToKICAgICAgICAgICAgZm9yIGsgaW4gc2VsZi5fY2xpY2tfY2FuZGlkYXRlcyhncmlkKToKICAgICAgICAgICAgICAgIGlmIGsgbm90IGluIG5vZGUudHJpZWQ6CiAgICAgICAgICAgICAgICAgICAgbm9kZS5wZW5kaW5nLmFwcGVuZChrKQoKICAgICMgLS0tLS0tLS0tLSBncmFmbyAtLS0tLS0tLS0tCgogICAgZGVmIF9yZWNvcmRfZWRnZShzZWxmLCBzcmM6IGludCwgYWN0aW9uOiB0dXBsZVtpbnQsIGludCwgaW50XSwgZHN0OiBpbnQpIC0+IE5vbmU6CiAgICAgICAgaWYgKHNyYywgYWN0aW9uKSBub3QgaW4gc2VsZi5fZWRnZXM6CiAgICAgICAgICAgIHNlbGYuX2VkZ2VzWyhzcmMsIGFjdGlvbildID0gZHN0CiAgICAgICAgICAgIHNlbGYuX2Fkai5zZXRkZWZhdWx0KHNyYywgW10pLmFwcGVuZCgoYWN0aW9uLCBkc3QpKQoKICAgIGRlZiBfYmZzX3RvX3BlbmRpbmcoc2VsZiwgc3RhcnQ6IGludCkgLT4gT3B0aW9uYWxbbGlzdFt0dXBsZVtpbnQsIGludCwgaW50XV1dOgogICAgICAgICIiIkNhbWlubyBtw6FzIGNvcnRvIChlbiBhcmlzdGFzIGNvbm9jaWRhcykgaGFzdGEgdW4gbm9kbyBjb24gcGVuZGllbnRlcy4iIiIKICAgICAgICBzZWVuID0ge3N0YXJ0fQogICAgICAgIHE6IGRlcXVlW3R1cGxlW2ludCwgbGlzdFt0dXBsZVtpbnQsIGludCwgaW50XV1dXSA9IGRlcXVlKFsoc3RhcnQsIFtdKV0pCiAgICAgICAgd2hpbGUgcToKICAgICAgICAgICAga2V5LCBwYXRoID0gcS5wb3BsZWZ0KCkKICAgICAgICAgICAgbm9kZSA9IHNlbGYuX25vZGVzLmdldChrZXkpCiAgICAgICAgICAgIGlmIG5vZGUgYW5kIG5vZGUucGVuZGluZyBhbmQga2V5ICE9IHN0YXJ0OgogICAgICAgICAgICAgICAgcmV0dXJuIHBhdGgKICAgICAgICAgICAgaWYgbGVuKHBhdGgpID49IDYwOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZm9yIGFjdGlvbiwgZHN0IGluIHNlbGYuX2Fkai5nZXQoa2V5LCBbXSk6CiAgICAgICAgICAgICAgICBpZiBkc3Qgbm90IGluIHNlZW46CiAgICAgICAgICAgICAgICAgICAgc2Vlbi5hZGQoZHN0KQogICAgICAgICAgICAgICAgICAgIHEuYXBwZW5kKChkc3QsIHBhdGggKyBbYWN0aW9uXSkpCiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICAjIC0tLS0tLS0tLS0gQVBJIC0tLS0tLS0tLS0KCiAgICBkZWYgY2hvb3NlKAogICAgICAgIHNlbGYsCiAgICAgICAgZ3JpZDogbnAubmRhcnJheSwKICAgICAgICBzdGF0ZTogc3RyLAogICAgICAgIGxldmVsc19jb21wbGV0ZWQ6IGludCwKICAgICAgICBhdmFpbGFibGVfYWN0aW9uczogbGlzdFtpbnRdLAogICAgKSAtPiB0dXBsZVtpbnQsIGludCwgaW50XToKICAgICAgICAiIiJEZXZ1ZWx2ZSAoYWN0aW9uX2lkLCB4LCB5KTsgeD15PS0xIHBhcmEgYWNjaW9uZXMgc2ltcGxlcy9SRVNFVC4iIiIKICAgICAgICBzZWxmLmFjdGlvbnNfdGFrZW4gKz0gMQogICAgICAgIGlmIHNlbGYuYWN0aW9uc190YWtlbiA+IHNlbGYubWF4X2FjdGlvbnM6CiAgICAgICAgICAgIHNlbGYuZG9uZSA9IFRydWUKCiAgICAgICAgIyAtLS0gZGlnZXJpciBlbCByZXN1bHRhZG8gZGUgbGEgYWNjacOzbiBhbnRlcmlvciAtLS0KICAgICAgICBpZiBzZWxmLl9sYXN0X2dyaWQgaXMgbm90IE5vbmUgYW5kIHNlbGYuX2xhc3RfYWN0aW9uIGlzIG5vdCBOb25lOgogICAgICAgICAgICBjaGFuZ2VkID0gYm9vbCgoc2VsZi5fbGFzdF9ncmlkICE9IGdyaWQpLmFueSgpKQogICAgICAgICAgICBzZWxmLl9sZWFybl9jb3VudGVyX21hc2soc2VsZi5fbGFzdF9ncmlkLCBncmlkKQogICAgICAgICAgICBhaWQsIGF4LCBheSA9IHNlbGYuX2xhc3RfYWN0aW9uCiAgICAgICAgICAgIGlmIGFpZCBpbiBzZWxmLl9hY3Rfc3RhdHM6CiAgICAgICAgICAgICAgICBzZWxmLl9hY3Rfc3RhdHNbYWlkXVsxXSArPSAxCiAgICAgICAgICAgICAgICBzZWxmLl9hY3Rfc3RhdHNbYWlkXVswXSArPSBpbnQoY2hhbmdlZCkKICAgICAgICAgICAgICAgIHNlbGYuX2FjdF9zdGF0c1thaWRdWzJdICs9IGludChzZWxmLl9rZXkoZ3JpZCkgbm90IGluIHNlbGYuX25vZGVzKQogICAgICAgICAgICBpZiBhaWQgPT0gNjoKICAgICAgICAgICAgICAgIHNpZyA9IHNlbGYuX3NpZ19hdChzZWxmLl9sYXN0X2dyaWQsIGF4LCBheSkKICAgICAgICAgICAgICAgIGlmIHNpZyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBpZiBjaGFuZ2VkIG9yIGxldmVsc19jb21wbGV0ZWQgIT0gc2VsZi5fbGFzdF9sZXZlbHM6CiAgICAgICAgICAgICAgICAgICAgICAgIHNlbGYuX2VmZl9zaWdzLmFkZChzaWcpCiAgICAgICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5fZGVhZF9zaWdzW3NpZ10gPSBzZWxmLl9kZWFkX3NpZ3MuZ2V0KHNpZywgMCkgKyAxCgogICAgICAgIGlmIGxldmVsc19jb21wbGV0ZWQgPiBzZWxmLl9sYXN0X2xldmVsczoKICAgICAgICAgICAgIyBuaXZlbCBudWV2bzogbG8gaW5lcnRlIGRlIGFudGVzIHB1ZWRlIHNlciBsYSBjbGF2ZSBhaG9yYQogICAgICAgICAgICBzZWxmLl9kZWFkX3NpZ3MuY2xlYXIoKQogICAgICAgICAgICBzZWxmLl9lZmZfc2lncy5jbGVhcigpCiAgICAgICAgICAgIHNlbGYuX3JlcGxheS5jbGVhcigpCiAgICAgICAgICAgIHNlbGYuX3JlcGxheV90YXJnZXQgPSBOb25lCiAgICAgICAgc2VsZi5fbGFzdF9sZXZlbHMgPSBsZXZlbHNfY29tcGxldGVkCgogICAgICAgIGtleSA9IHNlbGYuX2tleShncmlkKQogICAgICAgIGlmIHNlbGYuX2xhc3Rfa2V5IGlzIG5vdCBOb25lIGFuZCBzZWxmLl9sYXN0X2FjdGlvbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgc2VsZi5fcmVjb3JkX2VkZ2Uoc2VsZi5fbGFzdF9rZXksIHNlbGYuX2xhc3RfYWN0aW9uLCBrZXkpCgogICAgICAgICMgLS0tIGdhbWUgb3ZlciAvIG5vdCBwbGF5ZWQgLS0tCiAgICAgICAgaWYgc3RhdGUgaW4gKCJOT1RfUExBWUVEIiwgIkdBTUVfT1ZFUiIpOgogICAgICAgICAgICBzZWxmLl9jb25zZWN1dGl2ZV9yZXNldHMgKz0gMQogICAgICAgICAgICBpZiBzZWxmLl9jb25zZWN1dGl2ZV9yZXNldHMgPj0gUkVTRVRfTE9PUF9CUkVBSzoKICAgICAgICAgICAgICAgIHNlbGYuZG9uZSA9IFRydWUKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2NvbW1pdChncmlkLCBrZXksIFJFU0VUX0tFWSkKICAgICAgICBzZWxmLl9jb25zZWN1dGl2ZV9yZXNldHMgPSAwCgogICAgICAgICMgLS0tIHJlcGxheSBlbiBjdXJzbyAodmVyaWZpY2FuZG8gZGV0ZXJtaW5pc21vKSAtLS0KICAgICAgICBpZiBzZWxmLl9yZXBsYXk6CiAgICAgICAgICAgIGlmIHNlbGYuX3JlcGxheV90YXJnZXQgaXMgbm90IE5vbmUgYW5kIGtleSAhPSBzZWxmLl9yZXBsYXlfdGFyZ2V0OgogICAgICAgICAgICAgICAgc2VsZi5fcmVwbGF5LmNsZWFyKCkgICMgZWwgbXVuZG8gbm8gc2lndWnDsyBlbCBncmFmbzogYWJvcnRhciByZXBsYXkKICAgICAgICAgICAgICAgIHNlbGYuX3JlcGxheV90YXJnZXQgPSBOb25lCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBhY3Rpb24gPSBzZWxmLl9yZXBsYXkucG9wbGVmdCgpCiAgICAgICAgICAgICAgICBzZWxmLl9yZXBsYXlfdGFyZ2V0ID0gc2VsZi5fZWRnZXMuZ2V0KChrZXksIGFjdGlvbikpCiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fY29tbWl0KGdyaWQsIGtleSwgYWN0aW9uKQoKICAgICAgICBub2RlID0gc2VsZi5fbm9kZXMuZ2V0KGtleSkKICAgICAgICBpZiBub2RlIGlzIE5vbmU6CiAgICAgICAgICAgIG5vZGUgPSBfTm9kZSgpCiAgICAgICAgICAgIHNlbGYuX25vZGVzW2tleV0gPSBub2RlCiAgICAgICAgICAgIHNlbGYuX2ZpbGxfcGVuZGluZyhub2RlLCBncmlkLCBhdmFpbGFibGVfYWN0aW9ucykKCiAgICAgICAgaWYgbm9kZS5wZW5kaW5nOgogICAgICAgICAgICBhY3Rpb24gPSBub2RlLnBlbmRpbmcucG9wbGVmdCgpCiAgICAgICAgICAgIG5vZGUudHJpZWQuYWRkKGFjdGlvbikKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2NvbW1pdChncmlkLCBrZXksIGFjdGlvbikKCiAgICAgICAgIyBub2RvIGFnb3RhZG86IEJGUyBhbCBub2RvIHBlbmRpZW50ZSBtw6FzIGNlcmNhbm8KICAgICAgICBwYXRoID0gc2VsZi5fYmZzX3RvX3BlbmRpbmcoa2V5KQogICAgICAgIGlmIHBhdGg6CiAgICAgICAgICAgIHNlbGYuX3JlcGxheSA9IGRlcXVlKHBhdGgpCiAgICAgICAgICAgIGFjdGlvbiA9IHNlbGYuX3JlcGxheS5wb3BsZWZ0KCkKICAgICAgICAgICAgc2VsZi5fcmVwbGF5X3RhcmdldCA9IHNlbGYuX2VkZ2VzLmdldCgoa2V5LCBhY3Rpb24pKQogICAgICAgICAgICByZXR1cm4gc2VsZi5fY29tbWl0KGdyaWQsIGtleSwgYWN0aW9uKQoKICAgICAgICAjIGdyYWZvIGFsY2FuemFibGUgYWdvdGFkbzogcmVpbmljaW8gRElWRVJTSUZJQ0FETy4gRW4ganVlZ29zIGRldGVybWluaXN0YXMsCiAgICAgICAgIyByZS1leHBsb3JhciBjb24gZWwgbWlzbW8gb3JkZW4gcmVwZXRpcsOtYSBsYSB0cmF5ZWN0b3JpYTsgc3ViaW1vcyBlbCBzYWx0IHBhcmEKICAgICAgICAjIGRlbnNpZmljYXIgY2xpY2tzIHkgcGVydHVyYmFyIGVsIG9yZGVuLCB5IGFicmltb3MgZGUgbnVldm8gbGEgZXhwbG9yYWNpw7NuCiAgICAgICAgIyAob2x2aWRhbW9zIHBlbmRpbmcvdHJpZWQ7IGNvbnNlcnZhbW9zIGRlYWQvZWZmIHNpZ3MgeSBsYSBtw6FzY2FyYSBhcHJlbmRpZGEpLgogICAgICAgIHNlbGYuX2V4aGF1c3RlZF9yZXNldHMgKz0gMQogICAgICAgIHNlbGYuX3NhbHQgKz0gMQogICAgICAgIHNlbGYuX25vZGVzLmNsZWFyKCkKICAgICAgICBzZWxmLl9lZGdlcy5jbGVhcigpCiAgICAgICAgc2VsZi5fYWRqLmNsZWFyKCkKICAgICAgICBzZWxmLl9yZXBsYXkuY2xlYXIoKQogICAgICAgIHNlbGYuX3JlcGxheV90YXJnZXQgPSBOb25lCiAgICAgICAgaWYgc2VsZi5fZXhoYXVzdGVkX3Jlc2V0cyA+PSBNQVhfRVhIQVVTVEVEX1JFU0VUUzoKICAgICAgICAgICAgc2VsZi5kb25lID0gVHJ1ZQogICAgICAgIHJldHVybiBzZWxmLl9jb21taXQoZ3JpZCwga2V5LCBSRVNFVF9LRVkpCgogICAgZGVmIF9zaWdfYXQoc2VsZiwgZ3JpZDogbnAubmRhcnJheSwgeDogaW50LCB5OiBpbnQpIC0+IE9wdGlvbmFsW3R1cGxlW2ludCwgaW50LCBib29sXV06CiAgICAgICAgY291bnRzID0gbnAuYmluY291bnQoZ3JpZC5yYXZlbCgpLCBtaW5sZW5ndGg9MTYpCiAgICAgICAgYmFja2dyb3VuZCA9IGludChjb3VudHMuYXJnbWF4KCkpCiAgICAgICAgaWYgbm90ICgwIDw9IHggPCBHUklEIGFuZCAwIDw9IHkgPCBHUklEKSBvciBncmlkW3ksIHhdID09IGJhY2tncm91bmQ6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgZm9yIG8gaW4gY29ubmVjdGVkX2NvbXBvbmVudHMoZ3JpZCwgYmFja2dyb3VuZCk6CiAgICAgICAgICAgIHkwLCB4MCwgeTEsIHgxID0gb1siYmJveCJdCiAgICAgICAgICAgIGlmIHkwIDw9IHkgPD0geTEgYW5kIHgwIDw9IHggPD0geDE6CiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fY2xpY2tfc2lnKG8pCiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBkZWYgX2NvbW1pdChzZWxmLCBncmlkOiBucC5uZGFycmF5LCBrZXk6IGludCwgYWN0aW9uOiB0dXBsZVtpbnQsIGludCwgaW50XSkgLT4gdHVwbGVbaW50LCBpbnQsIGludF06CiAgICAgICAgc2VsZi5fbGFzdF9ncmlkID0gZ3JpZC5jb3B5KCkKICAgICAgICBzZWxmLl9sYXN0X2tleSA9IGtleQogICAgICAgIHNlbGYuX2xhc3RfYWN0aW9uID0gYWN0aW9uCiAgICAgICAgcmV0dXJuIGFjdGlvbgo="), ("hybrid_prelude", "IiIiUHJlbHVkaW8gZGUgZXhwbG9yYWNpw7NuOiBlbCBleHBsb3JhZG9yIENQVSBqdWVnYSBBTlRFUyBxdWUgZWwgTExNLCBlbiBjYWRhIGp1ZWdvLgoKUE9SIFFVRSAobWVkaWRvLCBkb2NzL0RFU0lHTi5tZCDCpzguMjIpLiBTb2JyZSBsb3MgMjUganVlZ29zIGxvY2FsZXMsIGNvbiBsYSBjb25maWcKZGVzcGxlZ2FkYSB5IHLDqWdpbWVuIGRlIDIgaCBwb3IganVlZ286CgogICAgTExNICAgICAgICA5IG5pdmVsZXMgICAgIHNvbG8gZWwgTExNIGdhbmEgZW46IGFyMjUsIGJwMzUsIHNiMjYKICAgIGV4cGxvcmFkb3IgMTggbml2ZWxlcyAgICBzb2xvIGVsIGV4cGxvcmFkb3IgZW46IGxzMjAsIG0wcjAsIHNjMjUsIHNwODAsIHRuMzYsIHR1OTMsIHZjMzMKICAgIHVuaW9uICAgICAgMjEgbml2ZWxlcwoKTGFzIGRvcyBjYXBhY2lkYWRlcyBzb24gZGlzdGludGFzIHkgZW4gcGFydGUgZGlzanVudGFzOiBicDM1IHkgc2IyNiBzb24ganVlZ29zIGRvbmRlCmVsIGV4cGxvcmFkb3IgZGEgQ0VSTyBpbmNsdXNvIGNvbiA0MC4wMDAgYWNjaW9uZXMsIHkgZWwgTExNIGxvcyByZXN1ZWx2ZTsgdHU5MyBlcyBsbwpjb250cmFyaW8uIExhIHVuaW9uIHZhbGUgMi4zeCBsbyBxdWUgZWwgTExNIHNvbG8uCgpQT1IgUVVFIFNFQ1VFTkNJQUwgWSBOTyBFTiBQQVJBTEVMTy4gSW50ZXJjYWxhciBhY2Npb25lcyBtaWVudHJhcyBlbCBMTE0gcGllbnNhIHJvbXBlCnRyZXMgY29zYXMgKMKnOC4yMSk6IHN1IGFjY2lvbiBzZSBhcGxpY2FyaWEgYSB1biB0YWJsZXJvIHF1ZSBubyB2aW8sIHN1cyBgaGlzdG9yeV9lbnRyaWVzYApjb250ZW5kcmlhbiBqdWdhZGFzIHF1ZSBubyBkZWNpZGlvIOKAlGVsIG1vZG8gZGUgZmFsbG8gcXVlIG1lZGltb3MgY29tbyBlbCBtYXMgZGFuaW5v4oCUIHkKaGFicmlhIGNhcnJlcmEgc29icmUgZWwgZmljaGVybyBkZSBlc3RhZG8uIENvcnJpZW5kbyBBTlRFUyBkZSBxdWUgYXJyYW5xdWUgbGEgc2VzaW9uIG5vCmV4aXN0ZSBuaW5ndW5vIGRlIGxvcyB0cmVzOiBlbCBleHBsb3JhZG9yIHRlcm1pbmEsIHkgZWwgTExNIGFicmUgc3UgaGlzdG9yaWFsIGRlc2RlIGVsCmVzdGFkbyByZXN1bHRhbnRlIGNvbW8gc2kgZnVlcmEgZWwgaW5pY2lhbC4KCkNPU1RFLiB+Mi4wMDAgYWNjaW9uZXMgc29icmUgZWwgZ2F0ZXdheSBhIH4wLjE1IHMgPSB+NSBtaW4gZGUgbG9zIDEzMiBkZSBsYSB2ZW50YW5hLgpFbCBMTE0gZXN0YSBsaW1pdGFkbyBwb3IgR1BVIChubyBwb3IgcmVsb2opIGRlbnRybyBkZSBzdSB2ZW50YW5hLCBhc2kgcXVlIHBpZXJkZSB+NCUgZGUKc3VzIHRva2Vucy4gQSBjYW1iaW8gYXJyYW5jYSBlbiBlbCBuaXZlbCBkb25kZSBsYSBidXNxdWVkYSBzZSBhdGFzY28g4oCUIHF1ZSBlcyBqdXN0byBkb25kZQpzdSBjb21wcmVuc2lvbiBoYWNlIGZhbHRhIHkgbGEgZnVlcnphIGJydXRhIHlhIG5vIGxsZWdhLgoKQ0FWRUFUIEhPTkVTVE8uIExvcyAyNSBqdWVnb3MgbG9jYWxlcyBTT04gbG9zIHB1YmxpY29zLCBzb2JyZSBsb3MgcXVlIGVzdGUgZXhwbG9yYWRvciBzZQphanVzdG8gZW4ganVsaW87IGVuIGVsIHNldCBvY3VsdG8gbWFyY28gMC4yNSBmcmVudGUgYWwgMC45NyBkZWwgaGFybmVzcy4gTGEgZXN0aW1hY2lvbgpwYXJhIGVsIG9jdWx0byAoMC4yNSB4IDY3JSBkaXNqdW50byB+ICswLjE3KSBlc3RhIGp1c3RvIGVuIGVsIGxpc3RvbiB5IGNvbiBiYXJyYXMgZGUKZXJyb3IgZ3JhbmRlcy4gRXN0ZSBtb2R1bG8gZXMgbGEgYXB1ZXN0YSwgbm8gdW5hIGNvbmNsdXNpb24uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRpbWUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKREVGQVVMVF9NQVhfQUNUSU9OUyA9IDIwMDAKREVGQVVMVF9NQVhfU0VDT05EUyA9IDQyMC4wICAgICAgICAjIHRvcGUgZHVybzogNyBtaW4gZGUgbGEgdmVudGFuYSBkZSAxMzIKTE9HX1BSRUZJWCA9ICJbaHlicmlkX3ByZWx1ZGVdIgoKCmRlZiBfZ3JpZF9mcm9tX3N0YXRlKHN0YXRlOiBBbnkpOgogICAgIiIiVWx0aW1vIGZyYW1lIHZpc2libGUgY29tbyBucC5uZGFycmF5LCBvIE5vbmUgc2kgbm8gc2UgcHVlZGUgbGVlci4iIiIKICAgIGltcG9ydCBudW1weSBhcyBucAoKICAgIHRyeToKICAgICAgICByYXcgPSBzdGF0ZS5yYXcuZnJhbWUKICAgICAgICBpZiBub3QgcmF3OgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHJldHVybiBucC5hc2FycmF5KHJhd1stMV0sIGR0eXBlPW5wLmludDgpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBOb25lCgoKZGVmIHJ1bl9wcmVsdWRlKGdhbWU6IEFueSwgbWF4X2FjdGlvbnM6IGludCA9IERFRkFVTFRfTUFYX0FDVElPTlMsCiAgICAgICAgICAgICAgICBtYXhfc2Vjb25kczogZmxvYXQgPSBERUZBVUxUX01BWF9TRUNPTkRTLAogICAgICAgICAgICAgICAgc2hvdWxkX3N0b3A6IEFueSA9IE5vbmUpIC0+IGRpY3Q6CiAgICAiIiJKdWVnYSBgZ2FtZWAgY29uIGVsIEdyYXBoRXhwbG9yZXIgaGFzdGEgYWdvdGFyIGFjY2lvbmVzLCB0aWVtcG8gbyBwcm9ncmVzby4KCiAgICBEZXZ1ZWx2ZSB1biByZXN1bWVuOyBOVU5DQSBsYW56YS4gQ3VhbHF1aWVyIGZhbGxvIGRlamEgZWwganVlZ28gY29tbyBlc3RhYmEgeQogICAgbGEgc2VzaW9uIGRlbCBMTE0gc2lndWUgc3UgY3Vyc28gbm9ybWFsIOKAlCBlbCBwZW9yIGNhc28gZXMgcXVlIGVzdGUgcHJlbHVkaW8gbm8KICAgIGhhZ2EgbmFkYSwgbm8gcXVlIHJvbXBhIGxhIHBhcnRpZGEuCiAgICAiIiIKICAgIGZyb20gYXJjMy5hZ2VudCBpbXBvcnQgR3JhcGhFeHBsb3JlciAgICAgICAgICAjIGVtYmViaWRvIGVuIGVsIG5vdGVib29rCiAgICBpbXBvcnQgYXJjZW5naW5lCgogICAgcmVzID0geyJhY2Npb25lcyI6IDAsICJuaXZlbGVzIjogMCwgIm1vdGl2byI6ICJvayJ9CiAgICB0cnk6CiAgICAgICAgcnVuID0gZ2V0YXR0cihnYW1lLCAiZ2FtZV9ydW4iLCBOb25lKQogICAgICAgIGlmIHJ1biBpcyBOb25lIG9yIGdldGF0dHIocnVuLCAic3RhdGUiLCBOb25lKSAhPSAicGxheWluZyI6CiAgICAgICAgICAgIHJlc1sibW90aXZvIl0gPSAiZWwganVlZ28gbm8gZXN0YSBlbiAncGxheWluZyciCiAgICAgICAgICAgIHJldHVybiByZXMKCiAgICAgICAgZ2FtZV9pZCA9IGdldGF0dHIocnVuLCAiZ2FtZV9pZCIsICI/IikKICAgICAgICBhZ2VudCA9IEdyYXBoRXhwbG9yZXIoZ2FtZV9pZCwgbWF4X2FjdGlvbnM9bWF4X2FjdGlvbnMgKyAxMCkKICAgICAgICB0MCA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICBuaXZlbGVzID0gaW50KGdhbWUuY3VycmVudF9zdGF0ZS5sZXZlbHNfY29tcGxldGVkKQoKICAgICAgICBmb3IgXyBpbiByYW5nZShtYXhfYWN0aW9ucyk6CiAgICAgICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgLSB0MCA+IG1heF9zZWNvbmRzOgogICAgICAgICAgICAgICAgcmVzWyJtb3Rpdm8iXSA9ICJ0b3BlIGRlIHRpZW1wbyIKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGlmIHNob3VsZF9zdG9wIGlzIG5vdCBOb25lIGFuZCBzaG91bGRfc3RvcCgpOgogICAgICAgICAgICAgICAgcmVzWyJtb3Rpdm8iXSA9ICJzdG9wX2V2ZW50IgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgZ2V0YXR0cihydW4sICJzdGF0ZSIsIE5vbmUpICE9ICJwbGF5aW5nIjoKICAgICAgICAgICAgICAgIHJlc1sibW90aXZvIl0gPSBmImVzdGFkbyB7Z2V0YXR0cihydW4sICdzdGF0ZScsICc/Jyl9IgogICAgICAgICAgICAgICAgYnJlYWsKCiAgICAgICAgICAgIHN0YXRlID0gZ2FtZS5jdXJyZW50X3N0YXRlCiAgICAgICAgICAgIGdyaWQgPSBfZ3JpZF9mcm9tX3N0YXRlKHN0YXRlKQogICAgICAgICAgICBpZiBncmlkIGlzIE5vbmU6CiAgICAgICAgICAgICAgICByZXNbIm1vdGl2byJdID0gInNpbiBmcmFtZSBsZWdpYmxlIgogICAgICAgICAgICAgICAgYnJlYWsKCiAgICAgICAgICAgIHZhbGlkYXMgPSBsaXN0KHN0YXRlLmF2YWlsYWJsZV9hY3Rpb25zIG9yIFtdKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAjIGF2YWlsYWJsZV9hY3Rpb25zIGRlbCBoYXJuZXNzIHNvbiBJTlRTIChbMCwxLDIsMyw0XSkgeSBlbAogICAgICAgICAgICAgICAgIyBleHBsb3JhZG9yIGxvcyBpbmRleGEgY29tbyB0YWxlcy4gUGFzYXJsb3MgY29tbyAiQUNUSU9OMCIuLi4KICAgICAgICAgICAgICAgICMgaGFjaWEgcXVlIG5vIGZvcm1hcmEgbmluZ3VuIGNhbmRpZGF0byB2YWxpZG8geSBkaWVyYSB2dWVsdGFzOgogICAgICAgICAgICAgICAgIyA2MDAgYWNjaW9uZXMgLT4gMCBuaXZlbGVzLiBQcm9iYWRvIGNvbnRyYSBlbCBHYW1lIHJlYWwsIHF1ZSBlcwogICAgICAgICAgICAgICAgIyBqdXN0byBsbyBxdWUgdW4gc3R1YiBubyBoYWJyaWEgZGV0ZWN0YWRvLgogICAgICAgICAgICAgICAgYWlkLCB4LCB5ID0gYWdlbnQuY2hvb3NlKGdyaWQsIHN0YXRlLnJhdy5zdGF0ZS52YWx1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpbnQoc3RhdGUubGV2ZWxzX2NvbXBsZXRlZCksIGxpc3QodmFsaWRhcykpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICByZXNbIm1vdGl2byJdID0gImVsIGV4cGxvcmFkb3IgZmFsbG8gYWwgZWxlZ2lyIgogICAgICAgICAgICAgICAgYnJlYWsKCiAgICAgICAgICAgICMgZWwgaGFybmVzcyBSRUNIQVpBIGFjY2lvbmVzIGZ1ZXJhIGRlIGF2YWlsYWJsZV9hY3Rpb25zOiBmaWx0cmFyIGFxdWkKICAgICAgICAgICAgIyBldml0YSBxdWUgdW5hIGV4Y2VwY2lvbiBhYm9ydGUgZWwgcHJlbHVkaW8gZW50ZXJvCiAgICAgICAgICAgIGlmIGFpZCBub3QgaW4gdmFsaWRhczoKICAgICAgICAgICAgICAgIGFsdGVybmF0aXZhID0gbmV4dCgodiBmb3IgdiBpbiB2YWxpZGFzIGlmIHYgIT0gMCksIE5vbmUpCiAgICAgICAgICAgICAgICBpZiBhbHRlcm5hdGl2YSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHJlc1sibW90aXZvIl0gPSAic2luIGFjY2lvbmVzIHZhbGlkYXMiCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGFpZCwgeCwgeSA9IGFsdGVybmF0aXZhLCB4LCB5CgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBhY3Rpb25faWQgPSBhcmNlbmdpbmUuR2FtZUFjdGlvbi5mcm9tX2lkKGFpZCkKICAgICAgICAgICAgICAgIGRhdGEgPSB7IngiOiBpbnQoeCksICJ5IjogaW50KHkpfSBpZiBhY3Rpb25faWQuaXNfY29tcGxleCgpIGVsc2Uge30KICAgICAgICAgICAgICAgIG51ZXZvID0gZ2FtZS5leGVjdXRlX2FjdGlvbigKICAgICAgICAgICAgICAgICAgICBhcmNlbmdpbmUuQWN0aW9uSW5wdXQoaWQ9YWN0aW9uX2lkLCBkYXRhPWRhdGEpKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgIyB1bmEgYWNjaW9uIHJlY2hhemFkYSBubyBkZWJlIHR1bWJhciBlbCBwcmVsdWRpbwogICAgICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgICAgIHJlc1siYWNjaW9uZXMiXSArPSAxCiAgICAgICAgICAgIG5pdmVsZXMgPSBtYXgobml2ZWxlcywgaW50KG51ZXZvLmxldmVsc19jb21wbGV0ZWQpKQogICAgICAgICAgICBpZiBnZXRhdHRyKG51ZXZvLCAid29uIiwgRmFsc2UpOgogICAgICAgICAgICAgICAgcmVzWyJtb3Rpdm8iXSA9ICJqdWVnbyBnYW5hZG8iCiAgICAgICAgICAgICAgICBicmVhawoKICAgICAgICByZXNbIm5pdmVsZXMiXSA9IG5pdmVsZXMKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgICAgICAgICAgICAgICAgICAgICAjIG5vcWE6IEJMRTAwMSDigJQgZGVncmFkYXIgc2llbXByZQogICAgICAgIHJlc1sibW90aXZvIl0gPSBmImV4Y2VwY2lvbiB7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iCiAgICByZXR1cm4gcmVzCg==")):
        _m = _types.ModuleType("arc3." + _n)
        _m.__package__ = "arc3"
        exec(compile(_b64.b64decode(_s).decode("utf-8"), "arc3/" + _n + ".py", "exec"),
             _m.__dict__)
        _sys.modules["arc3." + _n] = _m
        setattr(_pkg, _n, _m)
    from arc3.hybrid_prelude import run_prelude as _run_prelude
    import inference.framework.solver as _slv
    _HY_ACC = int(os.environ.get("HYBRID_ACTIONS", "12000"))
    _HY_SEC = float(os.environ.get("HYBRID_SECONDS", "420.0"))
    _orig_play = _slv._HarnessGameSession.play

    def _play_con_preludio(self):
        try:
            _r = _run_prelude(self.game, max_actions=_HY_ACC, max_seconds=_HY_SEC,
                              should_stop=self.should_stop)
            print(f"[hybrid] {getattr(self.game.game_run,'game_id','?')}: "
                  f"{_r['acciones']} acciones, nivel {_r['niveles']}, {_r['motivo']}",
                  flush=True)
        except Exception as _exc:   # el preludio NUNCA debe impedir jugar
            print(f"[hybrid] preludio fallo, sigo stock: {type(_exc).__name__}: {_exc}",
                  flush=True)
        return _orig_play(self)

    _slv._HarnessGameSession.play = _play_con_preludio
    print(f"HYBRID_PRELUDE installed: {_HY_ACC} acciones / {_HY_SEC}s por juego")
except Exception as exc:
    print(f"[hybrid_prelude] injection failed, running stock: {type(exc).__name__}: {exc}")

import arc_agi, taaf.game_api
def games_offline(d):
    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=d)
    ar = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=d)
    return [taaf.game_api.GameAPI(env_name=e.game_id, arcade_spec=spec) for e in ar.available_environments]
def games_comp():
    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.COMPETITION,
                                    arc_base_url=os.environ["ARC_BASE_URL"], environments_dir="")
    ar = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.COMPETITION,
                        arc_base_url=spec.arc_base_url, environments_dir="")
    return [taaf.game_api.GameAPI(env_name=e.game_id, arcade_spec=spec) for e in ar.available_environments]

soft_end = None
if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY","test-key-123")
    os.environ.setdefault("ARC_BASE_URL","http://gateway:8001/")
    dl = time.monotonic()+600
    while time.monotonic()<dl:
        try:
            with urlopen(os.environ["ARC_BASE_URL"]+"api/games", timeout=10) as r:
                if r.status<500: break
        except Exception: pass
        time.sleep(5)
    bm.games = games_comp()
else:
    bm.games = games_offline(str(COMP_ROOT/"environment_files"))
    soft_end = datetime.fromtimestamp(NOTEBOOK_START)+timedelta(minutes=OFFLINE_SOFT_MIN)

import pandas as pd
pd.DataFrame([["1_0","1",True,1]], columns=["row_id","game_id","end_of_game","score"]).to_parquet(WORKING/"submission.parquet", index=False)

try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
finally:
    for c in json.loads((BUNDLE/"teardown_commands.json").read_text()):
        subprocess.run(c, shell=True, check=False, cwd=WORKING, env=cmd_env())
print("run terminado")
